In [ ]:
from pyspark.sql.functions import col

from olist_silver.transformations import (
    is_valid_uuid,
    merge_into,
    null_invalid_uuid,
    with_processed_timestamp,
)

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_customers_table_name = dbutils.widgets.get("raw_olist_customers_table")

silver_schema = dbutils.widgets.get("silver_schema")
customers_table_name = dbutils.widgets.get("customers_table")

In [ ]:
raw_olist_customers_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_customers_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{customers_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{customers_table_name} (
            customerId STRING,
            customerUniqueId STRING,
            customerZipCodePrefix STRING,
            customerCity STRING,
            customerState STRING,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
customers_silver_df = with_processed_timestamp(
    raw_olist_customers_df.where(is_valid_uuid("customer_id"))
    .select(
        col("customer_id").cast("string").alias("customerId"),
        null_invalid_uuid("customer_unique_id").cast("string").alias("customerUniqueId"),
        col("customer_zip_code_prefix").cast("string").alias("customerZipCodePrefix"),
        col("customer_city").cast("string").alias("customerCity"),
        col("customer_state").cast("string").alias("customerState"),
    )
    .dropDuplicates(["customerId"])
)

In [ ]:
merge_into(
    spark,
    target=f"{catalog}.{silver_schema}.{customers_table_name}",
    source_view="customers_silver_view",
    keys=["customerId"],
    source_df=customers_silver_df,
)